In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/My Drive/Colab Notebooks/Baka töö')

Mounted at /content/drive


In [2]:
import pandas as pd
import random

In [3]:
'''
discs = pd.read_csv('pre_processing/discs_without_unknowns.csv')
manufacturers = pd.read_csv('pre_processing/manufacturers.csv')
'''

In [4]:
'''
# IDs for top 7 manufacturers
manufacturers_ids = [4, 13, 2, 17, 7, 15, 111]
quality_discs = []

for row in discs.iterrows():
  if row[1]['Manufacturer'] in manufacturers_ids:
    quality_discs.append(row[1])

# only discs from the top 7 manufacturers
quality_discs = pd.DataFrame(quality_discs)
quality_discs.to_csv('disc_suggestions/quality_discs.csv', index=False)
'''

In [5]:
quality_discs = pd.read_csv('disc_suggestions/quality_discs.csv')

# No need for understable putt&approach or midrange since we won't be suggesting those
understable_fairway_7 = []
understable_fairway_9 = []
understable_driver = [] # only suggest drivers with speeds 11 and 12

stable_putt_approach = []
stable_midrange = []
stable_fairway_7 = []
stable_fairway_9 = []
stable_driver = [] # only suggest drivers with speeds 11 and 12

overstable_approach = [] # for overstable approach we have putt&approach + speed 4 midranges
overstable_midrange = [] # That leaves only speeds 5 and 6 here
overstable_fairway_7 = []
overstable_fairway_9 = []
overstable_driver = [] # only suggest drivers with speeds 11 and 12

dictionary = {
    "understable_fairway_7" : understable_fairway_7,
    "understable_fairway_9" : understable_fairway_9,
    "understable_driver" : understable_driver,

    "stable_putt_approach" : stable_putt_approach,
    "stable_midrange" : stable_midrange,
    "stable_fairway_7" : stable_fairway_7,
    "stable_fairway_9" : stable_fairway_9,
    "stable_driver" : stable_driver,

    "overstable_approach" : overstable_approach,
    "overstable_midrange" : overstable_midrange,
    "overstable_fairway_7" : overstable_fairway_7,
    "overstable_fairway_9" : overstable_fairway_9,
    "overstable_driver" : overstable_driver
}

In [6]:
def resolve_disc_category(disc_id: int, category: str, speed: float, turn: float, stability: float) -> str:
    is_understable = (turn <= -2) or (-2 < turn < 0 and stability <= -1)
    is_overstable = (turn >= 0 and stability > 2) or (-2 < turn < 0 and stability >= 3)

    match category.lower():
        case "putt & approach":
            if is_understable:
                return "understable_putt_approach"
            if is_overstable:
                return "overstable_approach"
            return "stable_putt_approach"

        case "midrange":
            if is_understable:
                return "understable_midrange"
            if is_overstable:
                return "overstable_approach" if speed == 4 else "overstable_midrange"
            return "stable_midrange"

        case "fairway driver":
            return resolve_fairway_category(disc_id, speed, is_understable, is_overstable)

        case "distance driver":
            if is_understable:
                return "understable_driver"
            if is_overstable:
                return "overstable_driver"
            return "stable_driver"

        case _:
            raise ValueError(f"Disc with id={disc_id} has invalid type={category}")


def resolve_fairway_category(disc_id: int, speed: float, is_understable: bool, is_overstable: bool) -> str:
    speed_int = int(speed)
    if speed_int < 6 or speed_int > 9:
        raise ValueError(f"Disc with id={disc_id} has invalid speed={speed}")

    if is_understable:
        return {
            6: "understable_fairway_6",
            7: "understable_fairway_7",
            8: "understable_fairway_8"
        }.get(speed_int, "understable_fairway_9")

    if is_overstable:
        return {
            6: "overstable_fairway_6",
            7: "overstable_fairway_7",
            8: "overstable_fairway_8"
        }.get(speed_int, "overstable_fairway_9")

    return {
        6: "stable_fairway_6",
        7: "stable_fairway_7",
        8: "stable_fairway_8"
    }.get(speed_int, "stable_fairway_9")


In [7]:
for index, row in quality_discs.iterrows():
    stability = row['Turn'] + row['Fade']
    turn = row['Turn']
    disc_type = row['Type']
    speed = row['Speed']

    flight_numbers = []
    flight_numbers.append(row['Speed'])
    flight_numbers.append(row['Glide'])
    flight_numbers.append(row['Turn'])
    flight_numbers.append(row['Fade'])

    data = (row['ID'], row['Name'], stability, flight_numbers)

    category = resolve_disc_category(row['ID'], disc_type, speed, turn, stability)
    if category in dictionary:
        dictionary[category].append(data)

In [8]:
columns = ['disc_id', 'name', 'stability', 'speed', 'glide', 'turn', 'fade']
base_path = 'disc_suggestions/suggestion_categories/'

for name, disc_list in dictionary.items():
    rows = []
    for disc_id, disc_name, stability, stats in disc_list:
        rows.append([disc_id, disc_name, stability] + stats)

    df = pd.DataFrame(rows, columns=columns)
    file_path = base_path + f"{name}.csv"
    df.to_csv(file_path, index=False)